In [ ]:
# import packages
import pandas as pd
import numpy as np
import json, requests, geopy, folium, matplotlib

from sklearn.cluster import KMeans

# Get Data

In [ ]:
# get the original data from the Wikipedia
Toronto_df = pd.read_html("https://en.wikipedia.org/w/index.php?title=List_of_postal_codes_of_Canada:_M&oldid=926306543")[0]

# drop the unassigned rows
Toronto_df = Toronto_df[Toronto_df['Borough'] != 'Not assigned']

# combine neighbourhood
Toronto_df['Neighbourhood'] = Toronto_df.groupby(['Postcode'])['Neighbourhood'].transform(lambda x: ', '.join(x))

# drop the duplicate rows and reset the index
Toronto_df = Toronto_df.drop_duplicates().sort_values(by=['Postcode']).reset_index(drop=True)

In [ ]:
# read the data from website: latitude and longitude
df_cor = pd.read_csv('https://cocl.us/Geospatial_data')

# get the new dataframe with the latitude and longitude
Toronto_df_geo = Toronto_df.merge(df_cor, left_on = 'Postcode', right_on = 'Postal Code').drop(['Postal Code'], axis=1)
Toronto_df_geo.columns = [i.lower() for i in Toronto_df_geo.columns]
Toronto_df_geo.head(3)

# Clustering Data Preparation

In [ ]:
# # get the latitude and longtitude of Toronto, Canada
# address = 'Toronto, Canada'

# geolocator = geopy.geocoders.Nominatim(user_agent = "ny_explorer")
# latitude, longitude = geolocator.geocode(address).latitude, geolocator.geocode(address).longitude
# print(f'The geograpical coordinate of Toronto are {latitude}, {longitude}.')

In [ ]:
# create map of Toronto using latitude and longitude values
map_manhattan = folium.Map(location=[Toronto_df_geo['latitude'].mean(), Toronto_df_geo['longitude'].mean()], zoom_start=11)

# add markers to map
for lat, lng, postcode, neighbourhood in zip(Toronto_df_geo['latitude'], Toronto_df_geo['longitude'], Toronto_df_geo['postcode'], Toronto_df_geo['neighbourhood']):
    folium.CircleMarker([lat, lng], 
                        radius = 5,
                        tooltip=f"{postcode}: {neighbourhood}",
                        color = 'blue', fill = True, fill_color = '#3186cc', fill_opacity = 0.7).add_to(map_manhattan)  
map_manhattan

In [ ]:
# Define Foursquare Credentials and Version
client_id = 'BCIYDZYRYJJ4LJCUQLMSAEHRC1TPNOSFD2U1ANFMM5XK0MQQ' # your Foursquare ID
client_secret = 'HXMVCGI4YO0ULUWC30XJOO3O01JKNERPGHBJSQ2ERC25YVF5' # your Foursquare Secret
version = '20180605' # Foursquare API version

In [ ]:
# Create a function to get all the neighborhoods in Toronto
def getNearbyVenues(names, latitudes, longitudes, radius = 1000, limit = 100, client_id = client_id, client_secret = client_secret, version = version):
    venues_list = []
    
    for name, lat, lng in zip(names, latitudes, longitudes):
            
        # create the API request URL
        url = 'https://api.foursquare.com/v2/venues/explore?&client_id={}&client_secret={}&v={}&ll={},{}&radius={}&limit={}'.format(
            client_id, 
            client_secret, 
            version, 
            lat, 
            lng, 
            radius, 
            limit)
            
        # make the GET request
        results = requests.get(url).json()["response"]['groups'][0]['items']
        
        # return only relevant information for each nearby venue
        venues_list.append([(name, lat, lng, 
                             v['venue']['name'], v['venue']['location']['lat'], v['venue']['location']['lng'], 
                             v['venue']['location']['distance'], v['venue']['categories'][0]['name']) for v in results])

    nearby_venues = pd.DataFrame([item for venue_list in venues_list for item in venue_list])
    nearby_venues.columns = ['postcode', 'latitude', 'longitude', 'venue', 'venue_latitude', 'venue_longitude', 'venue_distance', 'venue_category']
    
    return(nearby_venues)

In [ ]:
# create a new dataframe called Toronto_venues
Toronto_venues = getNearbyVenues(Toronto_df_geo['postcode'], Toronto_df_geo['latitude'], Toronto_df_geo['longitude'])

print(f'There are {len(Toronto_venues["venue_category"].unique())} uniques categories.')

In [ ]:
# get the dummy data grouped
Toronto_venues_dummy = pd.get_dummies(Toronto_venues['venue_category'])
Toronto_venues_dummy['postcode'] = Toronto_venues['postcode']

Toronto_venues_dummy_group = Toronto_venues_dummy.groupby('postcode').mean().reset_index()

In [ ]:
# combine geo data with venues dummy data
Toronto_combined = Toronto_df_geo.merge(Toronto_venues_dummy_group, on='postcode')
Toronto_combined.head(3)

# Clustering

In [ ]:
# set number of clusters
kclusters = 10

# set the data for running the clustering
Toronto_clustering_data = Toronto_combined[Toronto_combined.columns[6: ]]

# run k-means clustering
kmeans = KMeans(n_clusters = kclusters, random_state = 0, n_init = 10).fit(Toronto_clustering_data)

# get the cuslter label
Toronto_combined['cluster_label'] = kmeans.labels_

In [ ]:
# create map
map_clusters = folium.Map(location=[Toronto_combined['latitude'].mean(), Toronto_combined['longitude'].mean()], zoom_start=11)

# set color scheme for the clusters
x = np.arange(kclusters)
ys = [i + x + (i*x)**2 for i in range(kclusters)]
colors_array = matplotlib.cm.rainbow(np.linspace(0, 1, len(ys)))
rainbow = [matplotlib.colors.rgb2hex(i) for i in colors_array]

# add markers to the map
markers_colors = []
for lat, lon, postcode, cluster in zip(Toronto_combined['latitude'], Toronto_combined['longitude'], Toronto_combined['postcode'], Toronto_combined['cluster_label']):
    folium.CircleMarker([lat, lon], 
                        radius = 5, 
                        tooltip = f"{postcode}: Cluster {cluster}",
                        color = rainbow[cluster-1], fill = True, fill_color = rainbow[cluster-1], fill_opacity = 0.7).add_to(map_clusters)
       
map_clusters